# Clasificación de Preguntas Biomédicas con BioBERT
## Fine-tuning sobre PubMedQA
**Integrantes:** Juan Andrés Cuervo - Jerónimo Velásquez - Santiago López

**Dataset:** [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA)  
**Modelo base:** [BioBERT](https://huggingface.co/dmis-lab/biobert-base-cased-v1.2)  
**Tarea:** Sequence Classification

---

## Descripción del Problema

PubMedQA es una tarea de **Question Answering biomédico** que consiste en responder preguntas de investigación médica a partir de abstracts de PubMed. Dado el título de un paper científico formulado como pregunta y el abstract correspondiente, el modelo debe determinar si la evidencia presentada responde **yes** o **no** a la pregunta.

**Ejemplo:**
- **Pregunta:** *Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?*
- **Contexto:** Abstract del paper con la evidencia experimental
- **Respuesta:** `yes`

---

## Utilidad del Proyecto

Este tipo de sistema tiene aplicaciones reales en:
- **Apoyo a decisiones clínicas** — responder preguntas médicas basadas en evidencia científica
- **Minería de literatura biomédica** — automatizar la revisión de miles de papers
- **Sistemas de búsqueda semántica** — encontrar evidencia relevante para una pregunta clínica

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


## Descripción del Dataset

### PubMedQA

PubMedQA es un dataset de Question Answering biomédico construido a partir de abstracts de PubMed. Fue creado por investigadores de Carnegie Mellon University y publicado en 2019.

### Subsets utilizados

| Subset | Tamaño | Descripción | Uso en este proyecto |
|---|---|---|---|
| `pqa_artificial` | 211.269 | Etiquetas generadas automáticamente | Entrenamiento |
| `pqa_labeled` | 1.000 | Anotado manualmente por expertos médicos | Validación y Test |

### Campos del dataset

| Campo | Tipo | Descripción |
|---|---|---|
| `pubid` | int | ID único del paper en PubMed |
| `question` | string | Título del paper formulado como pregunta |
| `context` | dict | Abstract del paper dividido en párrafos |
| `long_answer` | string | Conclusión detallada del paper |
| `final_decision` | string | Etiqueta: `yes`, `no` o `maybe` |

### Tipo de problema

- **Tarea NLP:** Sequence Classification
- **Entrada:** Pregunta + Abstract concatenados con token `[SEP]`
- **Salida:** Etiqueta binaria `yes` (1) / `no` (0)
- **Dominio:** Literatura científica biomédica en inglés

### Decisión de diseño

Se eliminaron los ejemplos `maybe` de `pqa_labeled` para mantener consistencia con `pqa_artificial` que solo contiene etiquetas binarias. El dataset de entrenamiento fue balanceado con igual número de ejemplos `yes` y `no` para evitar que el modelo aprenda a predecir siempre la clase mayoritaria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate
from sklearn.metrics import classification_report

## Análisis Exploratorio de Datos (EDA)

### Distribución de clases

El dataset original de `pqa_artificial` presenta un fuerte desbalance de clases:
- **yes:** 196.144 ejemplos (92.8%)
- **no:** 15.125 ejemplos (7.2%)

Para el entrenamiento se balancearon las clases tomando igual número de ejemplos de cada clase, eliminando el sesgo hacia la clase mayoritaria.

### Preprocesamiento del texto

Cada ejemplo combina la pregunta y el contexto en una sola secuencia de entrada:

[CLS] pregunta [SEP] contexto [SEP]

- **Longitud promedio:** ~1.472 caracteres por ejemplo
- **Truncamiento:** textos mayores a 512 tokens son truncados
- **Padding:** textos menores a 512 tokens son rellenados con ceros

### Splits finales

| Split | Ejemplos | Fuente | yes | no |
|---|---|---|---|---|
| Train | variable | `pqa_artificial` balanceado | 50% | 50% |
| Validation | 445 | `pqa_labeled` | 62% | 38% |
| Test | 445 | `pqa_labeled` | 62% | 38% |

### Tokenización

BioBERT utiliza **WordPiece tokenization** — palabras médicas largas o poco frecuentes se dividen en subpalabras:

- `hepatocellular` → `hepato` + `##cellular`
- `microalbuminuria` → `micro` + `##album` + `##inuria`

Esto permite al modelo manejar vocabulario médico especializado que no aparece en modelos de lenguaje general.

In [ ]:
dataset_artificial = load_dataset("qiaojin/PubMedQA", "pqa_artificial")
dataset_labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

print("pqa_artificial:")
print(dataset_artificial)
print("\npqa_labeled:")
print(dataset_labeled)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

pqa_artificial:
DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
})

pqa_labeled:
DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


In [ ]:
import pandas as pd

df_artificial = pd.DataFrame(dataset_artificial['train'])
df_labeled = pd.DataFrame(dataset_labeled['train'])

print("Clases en pqa_artificial:")
print(df_artificial['final_decision'].value_counts())

print("\nClases en pqa_labeled:")
print(df_labeled['final_decision'].value_counts())

Clases en pqa_artificial:
final_decision
yes    196144
no      15125
Name: count, dtype: int64

Clases en pqa_labeled:
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64


In [ ]:
df_yes = df_artificial[df_artificial['final_decision'] == 'yes'].sample(20000, random_state=42)
df_no = df_artificial[df_artificial['final_decision'] == 'no'].sample(15125, random_state=42)
df_artificial = pd.concat([df_yes, df_no]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train reducido: {len(df_artificial)} ejemplos")
print(df_artificial['final_decision'].value_counts())

Train reducido: 35125 ejemplos
final_decision
yes    20000
no     15125
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

df_labeled_filtered = df_labeled[df_labeled["final_decision"] != 'maybe'].reset_index(drop=True)

df_val, df_test = train_test_split(
    df_labeled_filtered,
    test_size=0.5,
    random_state=42,
    stratify=df_labeled_filtered['final_decision']
)

print(f"Train: {len(df_artificial)} ejemplos")
print(f"Validation: {len(df_val)} ejemplos")
print(f"Test: {len(df_test)} ejemplos")

print("\nDistribución validation:")
print(df_val['final_decision'].value_counts())

print("\nDistribución test:")
print(df_test['final_decision'].value_counts())

Train: 35125 ejemplos
Validation: 445 ejemplos
Test: 445 ejemplos

Distribución validation:
final_decision
yes    276
no     169
Name: count, dtype: int64

Distribución test:
final_decision
yes    276
no     169
Name: count, dtype: int64


In [ ]:
label2id = {"yes": 1, "no": 0}
id2label = {1: "yes", 0: "no"}

df_artificial['label'] = df_artificial['final_decision'].map(label2id)
df_val['label'] = df_val['final_decision'].map(label2id)
df_test['label'] = df_test['final_decision'].map(label2id)
print("Ejemplo de mapeo:")
print(df_artificial[['final_decision', 'label']].head())

Ejemplo de mapeo:
  final_decision  label
0             no      0
1            yes      1
2            yes      1
3            yes      1
4            yes      1


In [ ]:
def prepare_text(row):
  question = row['question']
  context = row['context']

  if isinstance(context, dict):
    context = " ".join(context.get('contexts', []))
  elif isinstance(context, list):
    context = " ".join(context)

  return str(question) + " [SEP] " + str(context)


df_artificial['text'] = df_artificial.apply(prepare_text, axis=1)
df_test['text'] = df_test.apply(prepare_text, axis=1)
df_val['text'] = df_val.apply(prepare_text, axis=1)

print("Ejemplo de texto de entrada:")
print(df_artificial['text'].iloc[0][:500])
print(f"\nLongitud promedio de textos (caracteres): {df_artificial['text'].str.len().mean():.0f}")

Ejemplo de texto de entrada:
Is polycythemia vera initiated by JAK2V617F mutation? [SEP] The somatic JAK2(V617F) mutation is seen in most polycythemia vera (PV) patients; however, it is not clear if JAK2(V617F) is the PV-initiating mutation. In order to examine this issue, we developed a novel real-time quantitative allele-specific PCR, in which allelic discrimination is enhanced by the synergistic effect of a mismatch in the -1 position, and a locked nucleic acid (LNA) nucleoside at the -2 position. Determination of alleli

Longitud promedio de textos (caracteres): 1475


## Modelo y Fine-tuning

### ¿Por qué BioBERT?

BioBERT es una versión especializada de BERT preentrenada sobre literatura biomédica. A diferencia de BERT genérico entrenado sobre Wikipedia y BookCorpus, BioBERT continuó su preentrenamiento sobre:

- **18,5 millones de abstracts de PubMed**
- **4,5 millones de artículos completos de PubMed Central (PMC)**

Esto le permite entender terminología médica especializada que BERT genérico desconoce.

### Técnica de Fine-tuning

El fine-tuning ajusta **todos los parámetros** del modelo sobre la tarea específica de clasificación. El proceso:

1. Se carga BioBERT con pesos preentrenados
2. Se agrega una cabeza de clasificación nueva (capa linear)
3. Se entrena sobre `pqa_artificial` balanceado
4. Se evalúa sobre `pqa_labeled` al final de cada época
5. Se guarda el checkpoint con mejor F1 en validación

### Función de pérdida

Se utilizó `CrossEntropyLoss` con pesos de clase para penalizar más los errores en la clase minoritaria cuando el dataset no está perfectamente balanceado.

### Métricas de evaluación

- **F1 weighted:** métrica principal, considera el desbalance entre clases
- **Accuracy:** porcentaje de predicciones correctas

In [ ]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texto_prueba = df_artificial['text'].iloc[0]

ejemplo_tokens = tokenizer(
    texto_prueba,
    truncation=True,
    max_length=512,
    padding='max_length',
    return_tensors='pt'
)

print(f"Modelo: {MODEL_NAME}")
print(f"Forma del input_ids: {ejemplo_tokens['input_ids'].shape}")
print(f"Tokens del primer ejemplo: {ejemplo_tokens['input_ids'].shape[1]}")

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Modelo: dmis-lab/biobert-base-cased-v1.2
Forma del input_ids: torch.Size([1, 512])
Tokens del primer ejemplo: 512


In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset

class PubMedDataset(TorchDataset):

  def __init__(self, df, tokenizer, max_length=512):
    self.df = df.reset_index(drop=True)
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    text = self.df['text'].iloc[idx]
    label = self.df['label'].iloc[idx]

    encoding = self.tokenizer(
        text,
        truncation=True,
        max_length=self.max_length,
        padding='max_length',
        return_tensors='pt'
    )

    return {
        'input_ids': encoding['input_ids'].squeeze(),
        'attention_mask': encoding['attention_mask'].squeeze(),
        'labels': torch.tensor(label, dtype=torch.long)
    }

train_dataset = PubMedDataset(df_artificial, tokenizer)
val_dataset = PubMedDataset(df_val, tokenizer)
test_dataset = PubMedDataset(df_test, tokenizer)

print(f"Train: {len(train_dataset)} ejemplos")
print(f"Validation: {len(val_dataset)} ejemplos")
print(f"Test: {len(test_dataset)} ejemplos")

Train: 35125 ejemplos
Validation: 445 ejemplos
Test: 445 ejemplos


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels_train = df_artificial['label'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1]),
    y=labels_train
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Peso clase 0 (no):  {class_weights_tensor[0]:.4f}")
print(f"Peso clase 1 (yes): {class_weights_tensor[1]:.4f}")

Peso clase 0 (no):  1.1612
Peso clase 1 (yes): 0.8781


In [ ]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)

  accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
  f1 = metric_f1.compute(predictions=predictions, references=labels, average='weighted')

  return {
      'accuracy': accuracy['accuracy'],
      'f1': f1['f1']
  }


In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer definido")

WeightedTrainer definido


## Experimentos y Comparación de Hiperparámetros

Se realizaron múltiples experimentos variando los hiperparámetros de entrenamiento para encontrar la configuración óptima del modelo.

### Hiperparámetros explorados

| Hiperparámetro | Descripción | Valores probados |
|---|---|---|
| `learning_rate` | Tamaño del paso de actualización de pesos | 3e-5 |
| `epochs` | Número de pasadas completas sobre el dataset | 5 |
| `warmup_steps` | Pasos de calentamiento del learning rate | 500 |
| `batch_size` | Ejemplos procesados simultáneamente | 16 |
| `train_size` | Tamaño del dataset de entrenamiento | 35125 |

### Observaciones

- **Más datos** fue el factor más influyente en la mejora del F1
- **Warmup steps** ayudó a estabilizar el entrenamiento inicial
- **Early stopping** evitó el overfitting detectado desde época 3
- El **validation loss** subió consistentemente después de época 2 en ambos experimentos, indicando overfitting moderado

In [ ]:
from transformers import EarlyStoppingCallback

results = []
models = [
    {
        "model_name": "dmis-lab/biobert-base-cased-v1.2",
        "learning_rate": 3e-5,
        "epochs": 5,
        "warmup_steps": 500,
        "weight_decay": 0.01,
        "batch_size": 16,
    },
]

for i, exp in enumerate(models):

    print(f"\n{'='*60}")
    print(f"Experimento {i+1}/{len(models)}: {exp['model_name']} | LR: {exp['learning_rate']} | Epochs: {exp['epochs']}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(exp['model_name'])
    train_dataset = PubMedDataset(df_artificial, tokenizer)
    val_dataset = PubMedDataset(df_val, tokenizer)
    test_dataset = PubMedDataset(df_test, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        exp['model_name'],
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"./results/exp_{i+1}",
        num_train_epochs=exp['epochs'],
        per_device_train_batch_size=exp['batch_size'],
        per_device_eval_batch_size=exp['batch_size'],
        learning_rate=exp['learning_rate'],
        weight_decay=exp['weight_decay'],
        warmup_steps=exp['warmup_steps'],
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=100,
        fp16=True,
        seed=42
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    test_results = trainer.evaluate(test_dataset)

    results.append({
        'modelo': exp['model_name'],
        'learning_rate': exp['learning_rate'],
        'epochs': exp['epochs'],
        'warmup_steps': exp['warmup_steps'],
        'weight_decay': exp['weight_decay'],
        'batch_size': exp['batch_size'],
        'test_f1': test_results['eval_f1'],
        'test_accuracy': test_results['eval_accuracy']
    })

    print(f"\nResultados experimento {i+1}:")
    print(f"F1: {test_results['eval_f1']:.4f}")
    print(f"Accuracy: {test_results['eval_accuracy']:.4f}")

    del model, trainer
    torch.cuda.empty_cache()


Experimento 1/1: dmis-lab/biobert-base-cased-v1.2 | LR: 3e-05 | Epochs: 5


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Resultados experimento 1:
F1: 0.7025
Accuracy: 0.6989


## BioBERT con Learning Rate 3e-5 y Batch Size 16

### Configuración

| Hiperparámetro | Valor |
|---|---|
| Modelo | dmis-lab/biobert-base-cased-v1.2 |
| Learning rate | 3e-5 |
| Epochs | 5 |
| Warmup steps | 500 |
| Weight decay | 0.01 |
| Batch size | 16 |

### Resultados por época

| Época | Training Loss | Validation Loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.2610 | 0.9912 | 0.6360 | 0.6321 |
| 2 | 0.1919 | 0.8721 | 0.6742 | 0.6738 |
| 3 | 0.1457 | 1.2751 | 0.7101 | 0.7127 |
| 4 | 0.0409 | 1.7483 | 0.7258 | 0.7296 |
| 5 | 0.0352 | 2.0842 | 0.7258 | 0.7292 |

### Resultados finales en Test

| Métrica | Valor |
|---|---|
| **F1** | **0.7025** |
| **Accuracy** | **0.6989** |

### Análisis

- El **Training Loss** disminuye consistentemente en todas las épocas — el modelo aprende correctamente sobre los datos de entrenamiento
- El **Validation Loss** sube desde época 3, indicando **overfitting** — el modelo memoriza el train pero no generaliza bien
- El F1 en validación se estabiliza en época 4 y 5 (0.7258) sin mejora significativa
- El modelo cargado al final es el de **época 4** por tener el mejor F1 en validación
